# Lecture 2 — Julia and VFI

**Computational Methods for Heterogeneous-Agent Macro**  
Jeffrey Sun

Solve the consumption-savings problem with no income shocks by Value Function Iteration (VFI).

Steps:
1. Define $V$ as a vector on a finite wealth grid.
2. Define the Bellman operator taking $V \mapsto \mathcal{T}V$ as a function taking a vector to a vector.
3. Repeatedly apply $\mathcal{T}\mathcal{T}\cdots\mathcal{T}\mathcal{T}V$ until convergence.

Notes:
- This code is not very optimized. I am using allocating functions, plain grid search, no Howard improvement, etc., etc.. I wrote it to be clear rather than fast.


### Environment


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots

# Core Code


### Utility and Log Grid

CRRA utility on `c > 0`, with the `γ = 1` case routed to `log`. A
log-spaced grid clusters points near the borrowing constraint.


In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length=N))

### Bellman Operator

Given a value function (guess) $V$, for each $b$ compute,
$$
TV(b) = \max_{b'} u(c) + \beta V(b')
$$
where
$$
\begin{align*}
    c &= b - b^\text{end}\\
    b^\text{end} &= \frac{b' - z}{R}
\end{align*}
$$

$$
TV(b) = \max_{b'} u(c) + \beta V(b'),
$$
$$
\begin{align*}
    c &= b - b^\text{end},\\
    b^\text{end} &= \frac{b' - z}{R}.
\end{align*}
$$

Here $z$ is the agent's productivity / human-capital level; it is a deterministic scalar in L02 and becomes stochastic in L03.


In [ ]:
"""
Apply Bellman operator V |-> TV, when V is a vector over the wealth grid and there are no income shocks.

"""
function bellman_operator(V, params)
    # Unpack parameters from `params`
    (; β, R, z, b_grid) = params

    # Compute saving levels `b_end` necessary to achieve each level of next-period wealth `b_next`
    b_end = (b_grid' .- z)./R

    return maximum(u.(b_grid .- b_end) .+ β.*V'; dims=2)
end

### Value Function Iteration

Iterate $V \mapsto \mathcal{T}V$ until the max-abs change is below `tol`.


In [ ]:
function solve_vfi(params; tol=1e-6, maxiter=2000, verbosity=0)
    # Make initial guess of V
    V = zeros(size(params.b_grid))

    Δ = Inf
    iters = 0
    while Δ >= tol
        # Update V
        TV = bellman_operator(V, params)

        # Calculate error
        Δ = maximum(abs.(TV .- V))

        # Update V
        V = TV
        iters = iters + 1

        # If iters > maxiter, then error.
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    # If verbosity >= 1, then print convergence message.
    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")

    return V
end

In [ ]:
b_grid = loggrid(0.05, 40.0, 400)
params = (;β=0.96, R=1.04, z=1.0, b_grid)

@show solve_vfi(params, verbosity=1)
;

## Solve and plot

V is increasing and concave.

With more iterations, the effect of the update gets smaller and smaller.


In [ ]:
b_grid = loggrid(0.05, 40.0, 400)
params = (;β=0.96, R=1.04, z=1.0, b_grid)

# Guess initial V
V = zeros(size(b_grid))
V_0 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_20 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_40 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_60 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_80 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_100 = copy(V)

plot(b_grid, hcat(V_0, V_20, V_40, V_60, V_80, V_100), label=["V_0" "V_20" "V_40" "V_60" "V_80" "V_100"])

### Policy Function

The policy function is the same operator as the Bellman operator, with
`argmax` in place of `max`:
$$
c^\star(b) \;=\; \arg\max_{c \in [0, b]} \; u(c) + \beta\, V\!\big(R(b - c) + z\big).
$$

$$
c^\star(b) \;=\; \arg\max_{c \in [0, b]} \; u(c) + \beta\, V\!\big(R(b - c) + z\big).
$$


In [ ]:
"""
Compute the policy index vector `c_ind` from V. For each row of b_grid,
returns the column index in b_grid of the optimal next-period wealth b'.
Same operator as `bellman_operator`, with `argmax` instead of `max`.

"""
function policy_function(V, params)
    (; β, R, z, b_grid) = params

    # same setup as bellman_operator
    b_end = (b_grid' .- z) ./ R

    # argmax over b' (columns) — pull out the column index for each row
    idx = argmax(u.(b_grid .- b_end) .+ β .* V'; dims=2)
    return vec(getindex.(idx, 2))::Vector{Int}
end


In [ ]:
# read the policy off the converged V — c_ind[i] is the chosen b'-index at b_grid[i]
c_ind = policy_function(V, params);


# Step-by-Step Explanation


### Utility and Log Grid


In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length=N))

In [ ]:
# Compute utility for a certain consumption level,
@show u(2.5)
@show u(3.0)
# for multiple possible consumption levels,
@show u.([1, 2, 3])
# for a whole grid of possible consumption levels.
@show c_grid = loggrid(0.1, 10.0, 100)
@show u.(c_grid)
;

### Bellman Operator


In [ ]:
"""
Apply Bellman operator V |-> TV, when V is a vector over the wealth grid and there are no income shocks.

"""
function bellman_operator(V, params)
    # Unpack parameters from `params`
    (; β, R, z, b_grid) = params

    # Compute saving levels `b_end` necessary to achieve each level of next-period wealth `b_next`
    b_end = (b_grid' .- z)./R

    return maximum(u.(b_grid .- b_end) .+ β.*V'; dims=2)
end

In [ ]:
b_grid = loggrid(0.05, 40.0, 400)
β=0.96
R=1.04
z=1.0

# Think of axis 2 (columns) as representing next-period wealth `b_next`.
# We can get the grid of `b_next` values by simply transposing b_grid.

# Note that b_grid is a COLUMN vector
b_grid

In [ ]:
# But b_grid' is a ROW vector
b_grid'

In [ ]:
z = 0.5

# Then, the end-of-period wealth necessary to achieve each `b_next` is given by,
@show b_end = (b_grid' .- z)./R
# Note that `b_end` is still a row vector, because every entry in b_end corresponds to a different level of b_next.

# Another way to think about it:
# b_next_grid = b_grid'
# b_end = (b_next_grid .- z)./R

# For each current level of wealth (row), a household must choose their next-period wealth `b_next` (column).
# The possible utilities are then going to be MATRIX, with a ROW for each possible b, and a COLUMN for each possible b_next.
u.(b_grid .- b_end)

In [ ]:
# Now let's solve the decision problem.
# Suppose we have some guess for V
V = ones(size(b_grid))

# We can define `V_next` to be the (guessed) value of having each next-period wealth `b_next`.
# The grid of `V_next` values for each b_next is simply V'.
# Then, the matrix of u + βV_next is,
u.(b_grid .- b_end) .+ β.*V'

In [ ]:
# The solution to the household's decision problem is simply,
maximum(u.(b_grid .- b_end) .+ β.*V'; dims=2)

In [ ]:
# What happens if we apply the Bellman operator over and over?
# Set up params
b_grid = loggrid(0.05, 40.0, 400)
params = (;β=0.96, R=1.04, z=1.0, b_grid)

# Guess V
V = zeros(size(b_grid))
res_mat = hcat(b_grid, V)

for i = 1:200
    V = bellman_operator(V, params)
    res_mat = hcat(res_mat, V)
end

res_mat

In [ ]:
# The initial guess doesn't change the limit that V converges to.
# Guess V
V = ones(size(b_grid))
res_mat = hcat(b_grid, V)

for i = 1:200
    V = bellman_operator(V, params)
    res_mat = hcat(res_mat, V)
end

res_mat

### Value Function Iteration (VFI)


In [ ]:
function solve_vfi(params; tol=1e-6, maxiter=2000, verbosity=0)
    # Make initial guess of V
    V = zeros(size(params.b_grid))

    Δ = Inf
    iters = 0
    while Δ >= tol
        # Update V
        TV = bellman_operator(V, params)

        # Calculate error
        Δ = maximum(abs.(TV .- V))

        # Update V
        V = TV
        iters = iters + 1

        # If iters > maxiter, then error.
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    # If verbosity >= 1, then print convergence message.
    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")

    return V
end